# 46 — Anchored-PRR strong-base substitution on the frozen 24-condition benchmark

## Question
Can the proposed future-supervised PRR framework improve further when its conventional retrieval branch is replaced by the much stronger **last-value-anchored L2** retriever?

This notebook tests a deliberately controlled variant, **Anchored-PRR (A-PRR)**:

\[
\mathcal C_{A\text{-}PRR}(q)
= \mathcal C^{A}_{100}(q) \cup \mathcal C^{emb}_{100}(q)
\rightarrow \text{future-supervised fusion reranking}
\rightarrow \text{Top-10}.
\]

where:
- `Anchored Top-100` uses endpoint-relative past shape, \(\tilde x_t=x_t-x_L\), and squared L2 distance;
- `Learned Top-100` reuses the **same frozen future-supervised dual encoder** from the existing full PRR stack;
- **only the fusion reranker is retrained from scratch** for this strong-base substitution;
- the training geometry, future-compatible target, listwise + pairwise loss, Phase-A model selection, and final refit all match the existing PRR recipe;
- test evaluation uses the exact frozen Exp33 manifests, same-channel admissible memory, `L=96`, memory stride `8`, and `Top-K=10`.

### Why freeze the learned encoder?
This is the cleanest first experiment: the learned future-supervised branch is held fixed while the conventional branch is changed from Pearson to Anchored L2. If A-PRR improves over Anchored L2, the gain cannot be attributed to retraining a different representation model.

### Anchored base score used by the residual reranker
Anchored L2 ranking itself is unchanged. For numerical stability in the additive residual score, the negative anchored MSE is standardized **query-wise over the full admissible memory**:

\[
s_A(q,i)=\frac{-d_A(q,i)-\mu_q}{\sigma_q+10^{-6}}.
\]

This is a monotone transformation for each query, so it produces exactly the same Anchored Top-100 ordering as raw anchored L2 distance.

### Primary comparisons
1. **A-PRR vs Anchored L2** — the key strong-base test.
2. **A-PRR vs current PRR** — does the stronger base improve the current framework?
3. **A-PRR vs Pearson** — contextual reference only.

No downstream forecaster experiment is run here. First decide from the 24 retrieval conditions whether the stronger-base PRR is worth propagating to the 120-condition frozen-forecaster study.

In [1]:
# Kernel guard — this experiment must run in the Stock Regime kernel.
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])

if sys.version_info < (3, 12):
    raise RuntimeError(
        "Wrong Jupyter kernel. Select 'Python 3.12 (Stock Regime)' / "
        "'Stock Regime' before running this notebook."
    )

print("[PASS] Stock Regime-compatible Python kernel detected")


Python executable: /data/envs/stock_regime/bin/python
Python version   : 3.12.13
[PASS] Stock Regime-compatible Python kernel detected


In [2]:
# Cell 1 — imports, reproducibility, and frozen-project discovery
from pathlib import Path
from dataclasses import dataclass
import os, re, gc, math, random, time, json, hashlib

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 700)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "This reranker-training experiment requires CUDA."

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)
EPS = 1e-8

manifest_candidates = [
    Path("/data/code/which_histories_matter_ext/results/exp33/final_eval_manifest"),
    Path("/data/code/stock_regime_retrieval/strong_forecaster/results/exp33/final_eval_manifest"),
]
for base in [Path("/data/code"), Path("/data")]:
    if base.exists():
        manifest_candidates += list(base.glob("*/results/exp33/final_eval_manifest"))
        manifest_candidates += list(base.glob("*/*/results/exp33/final_eval_manifest"))

MANIFEST_ROOT = next((p for p in manifest_candidates if p.exists()), None)
assert MANIFEST_ROOT is not None, "Exp33 frozen manifests were not found."

PROJECT_ROOT = MANIFEST_ROOT.parents[2]
OUT_ROOT = PROJECT_ROOT / "results" / "exp46_anchored_prr_strong_base"
CKPT_ROOT = OUT_ROOT / "rerankers"
CACHE_ROOT = OUT_ROOT / "test_parts"
for p in [OUT_ROOT, CKPT_ROOT, CACHE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("MANIFEST_ROOT:", MANIFEST_ROOT)
print("OUT_ROOT     :", OUT_ROOT)
print("GPU          :", torch.cuda.get_device_name(0))
print("torch        :", torch.__version__)

PROJECT_ROOT : /data/code/which_histories_matter_ext
MANIFEST_ROOT: /data/code/which_histories_matter_ext/results/exp33/final_eval_manifest
OUT_ROOT     : /data/code/which_histories_matter_ext/results/exp46_anchored_prr_strong_base
GPU          : NVIDIA A100-SXM4-80GB
torch        : 2.12.1+cu126


In [3]:
# Cell 2 — experiment configuration
@dataclass
class CFG:
    retrieval_seq_len: int = 96
    memory_stride: int = 8
    top_k: int = 10
    top_m_anchored: int = 100
    top_m_learned: int = 100
    union_max: int = 200

    # Existing PRR train-only geometry
    train_memory_fraction: float = 0.60
    train_query_stride: int = 4
    phase_a_train_fraction: float = 0.80
    screen_max_train_channels: int = 64
    screen_max_train_pairs: int = 6000

    emb_dim: int = 64
    enc_hidden: int = 128

    rerank_hidden: int = 384
    rerank_dropout: float = 0.1
    rerank_batch: int = 64
    rerank_lr: float = 1e-3
    rerank_wd: float = 1e-4
    rerank_max_epochs: int = 30
    rerank_patience: int = 6
    teacher_tau: float = 0.5
    pairwise_lambda: float = 1.0
    grad_clip: float = 5.0

    feature_clip: float = 8.0
    seed: int = 0
    query_batch: int = 64

cfg = CFG()

# One-click default: train/evaluate all 24 conditions, but resume from saved rerankers/parts.
RUN_TRAINING = True
RUN_EVALUATION = True
OVERWRITE_RERANKER = False
OVERWRITE_TEST_CACHE = False

# For a quick smoke test use, e.g. TARGETS=[("Weather",96)].
TARGETS = [(d,h) for d in DATASETS for h in HORIZONS]


def seed_everything(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)
print("Targets:", len(TARGETS))

Targets: 24


In [4]:
# Cell 3 — resolve the existing 24 full-PRR checkpoints (learned branch source)
REVIEWER_STACK_ROOT = PROJECT_ROOT / "artifacts" / "full_prr_training" / "full_stacks"

def legacy_stack_path(dataset, horizon):
    if dataset == "ETTh1":
        return (
            PROJECT_ROOT / "checkpoints"
            / "11_etth1_all_horizons_five_backbones_fixed_alpha01"
            / "full_stacks" / f"ETTh1_H{int(horizon)}_Full.pt"
        )
    return (
        PROJECT_ROOT / "checkpoints"
        / "12_multidataset_screening_itransformer_segmoe_fixed_alpha01"
        / "full_stacks" / f"{dataset}_H{int(horizon)}_screen.pt"
    )

def reproduced_stack_path(dataset, horizon):
    suffix = "Full" if dataset == "ETTh1" else "screen"
    return REVIEWER_STACK_ROOT / f"{dataset}_H{int(horizon)}_{suffix}.pt"

STACKS = {}
stack_rows = []
for d in DATASETS:
    for h in HORIZONS:
        a = reproduced_stack_path(d,h)
        b = legacy_stack_path(d,h)
        if a.is_file():
            chosen, source = a, "reproduced_full_prr"
        elif b.is_file():
            chosen, source = b, "original_full_prr"
        else:
            chosen, source = a, "missing"
        STACKS[(d,h)] = chosen
        stack_rows.append(dict(dataset=d,horizon=h,exists=chosen.is_file(),source=source,path=str(chosen)))

stack_inventory = pd.DataFrame(stack_rows)
display(stack_inventory)
assert len(stack_inventory)==24
assert stack_inventory.exists.all(), "Missing full-PRR checkpoints: run the full-PRR training/reproduction notebook first."
print("[PASS] frozen future-supervised encoder source = 24/24")

,dataset,horizon,exists,source,path
0,ETTh1,96,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...
1,ETTh1,192,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...
2,ETTh1,336,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...
3,ETTh1,720,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...
4,Weather,96,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...
5,Weather,192,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...
6,Weather,336,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...
7,Weather,720,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...
8,Electricity,96,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...
9,Electricity,192,True,original_full_prr,/data/code/which_histories_matter_ext/checkpoi...


[PASS] frozen future-supervised encoder source = 24/24


In [5]:
# Cell 4 — datasets and exact train-only normalization
EXPECTED_CHANNELS = {
    "ETTh1":7, "Weather":21, "Electricity":321,
    "Traffic":862, "Exchange":8, "Solar":137,
}

DATA_CANDIDATES = {
    "ETTh1":[
        "/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv",
        "/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv",
        "/data/dataset/ETT-small/ETTh1.csv", "/data/dataset/ETTh1.csv"],
    "Weather":[
        "/data/Time-Series-Library/dataset/weather/weather.csv",
        "/data/Time-Series-Library_v2/dataset/weather/weather.csv",
        "/data/dataset/weather/weather.csv", "/data/dataset/weather.csv"],
    "Electricity":[
        "/data/Time-Series-Library/dataset/electricity/electricity.csv",
        "/data/Time-Series-Library_v2/dataset/electricity/electricity.csv",
        "/data/dataset/electricity/electricity.csv", "/data/dataset/electricity.csv"],
    "Traffic":[
        "/data/Time-Series-Library/dataset/traffic/traffic.csv",
        "/data/Time-Series-Library_v2/dataset/traffic/traffic.csv",
        "/data/dataset/traffic/traffic.csv", "/data/dataset/traffic.csv"],
    "Exchange":[
        "/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv",
        "/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv",
        "/data/dataset/exchange_rate/exchange_rate.csv", "/data/dataset/exchange_rate.csv"],
    "Solar":[
        "/data/Time-Series-Library/dataset/Solar/solar_AL.txt",
        "/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt",
        "/data/dataset/Solar/solar_AL.txt", "/data/dataset/solar_AL.txt"],
}

def resolve_data(name):
    for s in DATA_CANDIDATES[name]:
        p = Path(s)
        if p.is_file():
            return p
    raise FileNotFoundError(name)

def clean_numeric(df):
    return (df.apply(pd.to_numeric,errors="coerce")
              .dropna(axis=1,how="all")
              .replace([np.inf,-np.inf],np.nan)
              .interpolate(axis=0,limit_direction="both")
              .ffill().bfill())

def load_dataset(name):
    p = resolve_data(name)
    if name == "Solar":
        df = pd.read_csv(p,header=None)
        if df.shape[1] == 1:
            df = pd.read_csv(p,header=None,sep=r"\s+")
        num = clean_numeric(df)
    else:
        df = pd.read_csv(p)
        date = next((c for c in df.columns if str(c).lower() in {"date","datetime","timestamp","time"}),None)
        val = df.drop(columns=[date]) if date is not None else df.copy()
        if name in {"Weather","Electricity"} and "OT" in val.columns:
            val = val[[c for c in val.columns if c!="OT"]+["OT"]]
        num = clean_numeric(val)

    raw = num.to_numpy(np.float32)
    assert raw.shape[1] == EXPECTED_CHANNELS[name], (name,raw.shape)
    n = len(raw)
    if name == "ETTh1":
        assert raw.shape == (17420,7)
        train_end, val_end = 8640, 11520
    else:
        train_end = int(0.70*n)
        val_end = n-int(0.20*n)

    mu = raw[:train_end].mean(0).astype(np.float32)
    sd = raw[:train_end].std(0,ddof=0).astype(np.float32)
    assert np.all(sd > 1e-6), (name, np.where(sd<=1e-6)[0][:20])
    z = ((raw-mu)/sd).astype(np.float32)
    return dict(name=name,path=p,z=z,n_time=n,n_channels=raw.shape[1],train_end=train_end,val_end=val_end)

DATA = {d:load_dataset(d) for d in DATASETS}
display(pd.DataFrame([dict(dataset=d,rows=DATA[d]["n_time"],channels=DATA[d]["n_channels"],
                           train_end=DATA[d]["train_end"],val_end=DATA[d]["val_end"],path=str(DATA[d]["path"]))
                      for d in DATASETS]))

,dataset,rows,channels,train_end,val_end,path
0,ETTh1,17420,7,8640,11520,/data/Time-Series-Library/dataset/ETT-small/ET...
1,Weather,52696,21,36887,42157,/data/Time-Series-Library/dataset/weather/weat...
2,Electricity,26304,321,18412,21044,/data/Time-Series-Library/dataset/electricity/...
3,Traffic,17544,862,12280,14036,/data/Time-Series-Library/dataset/traffic/traf...
4,Exchange,7588,8,5311,6071,/data/Time-Series-Library/dataset/exchange_rat...
5,Solar,52560,137,36792,42048,/data/Time-Series-Library/dataset/Solar/solar_...


In [6]:
# Cell 5 — architecture and exact feature helpers

def batch_pattern(x):
    x=np.asarray(x,np.float32)
    xc=x-x.mean(axis=-1,keepdims=True)
    n=np.linalg.norm(xc,axis=-1,keepdims=True)
    return np.where(n>EPS,xc/np.maximum(n,EPS),0.0).astype(np.float32)

def anchored_shape(x):
    x=np.asarray(x,np.float32)
    return (x-x[...,-1:]).astype(np.float32)

def anchored_similarity_z(qpast, mpast):
    """Query-wise standardized negative anchored MSE; ranking == raw anchored L2."""
    q=anchored_shape(qpast)
    m=anchored_shape(mpast)
    qn=np.sum(q*q,axis=1,keepdims=True)
    mn=np.sum(m*m,axis=1)[None,:]
    d2=np.maximum(qn+mn-2.0*(q@m.T),0.0)/q.shape[1]
    raw=(-d2).astype(np.float32)
    mu=raw.mean(axis=1,keepdims=True)
    sd=raw.std(axis=1,keepdims=True)
    return ((raw-mu)/(sd+1e-6)).astype(np.float32)

def context7(x):
    x=np.asarray(x,np.float32)
    short=max(8,cfg.retrieval_seq_len//4)
    m=x.mean(axis=-1); s=x.std(axis=-1)+EPS
    f1=(x[...,-1]-m)/s
    f2=(x[...,-short:].mean(axis=-1)-m)/s
    f3=(x[...,-1]-x[...,-short])/s
    f4=(x[...,-1]-x[...,0])/s
    df=np.diff(x,axis=-1); ds=np.diff(x[...,-short:],axis=-1)
    f5=(ds.std(axis=-1)+EPS)/(df.std(axis=-1)+EPS)
    t=np.linspace(-1.,1.,cfg.retrieval_seq_len,dtype=np.float32); t=t-t.mean()
    f6=(np.sum(t*(x-m[...,None]),axis=-1)/(np.sum(t*t)+EPS))/s
    a=x[...,:-1]; b=x[...,1:]
    a=a-a.mean(axis=-1,keepdims=True); b=b-b.mean(axis=-1,keepdims=True)
    f7=np.sum(a*b,axis=-1)/(np.sqrt(np.sum(a*a,axis=-1)*np.sum(b*b,axis=-1))+EPS)
    return np.stack([f1,f2,f3,f4,f5,f6,f7],axis=-1).astype(np.float32)

def extract_channel(z,ch,anchors,horizon):
    anchors=np.asarray(anchors,np.int64)
    pi=anchors[:,None]-cfg.retrieval_seq_len+np.arange(cfg.retrieval_seq_len)[None,:]
    fi=anchors[:,None]+np.arange(int(horizon))[None,:]
    past=z[pi,int(ch)].astype(np.float32)
    fut=z[fi,int(ch)].astype(np.float32)
    cur=z[anchors-1,int(ch)].astype(np.float32)
    return past,(fut-cur[:,None]).astype(np.float32)

class TemporalEncoder1D(nn.Module):
    def __init__(self,hidden=128,emb_dim=64):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv1d(1,hidden//2,5,stride=2,padding=2),nn.GELU(),
            nn.Conv1d(hidden//2,hidden,5,stride=2,padding=2),nn.GELU(),
            nn.Conv1d(hidden,hidden,3,stride=2,padding=1),nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj=nn.Linear(hidden,emb_dim)
    def forward(self,x):
        if x.ndim==2: x=x[:,None,:]
        return F.normalize(self.proj(self.net(x).squeeze(-1)),dim=-1)

def inverse_softplus(x):
    return math.log(math.exp(float(x))-1.0)

class WideFusionReranker(nn.Module):
    def __init__(self):
        super().__init__()
        in_dim=1+4*(7+cfg.emb_dim)  # same 285-D geometry as full PRR
        h=cfg.rerank_hidden
        self.net=nn.Sequential(
            nn.Linear(in_dim,h),nn.LayerNorm(h),nn.GELU(),nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h,h),nn.LayerNorm(h),nn.GELU(),nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h,1),
        )
        self.raw_alpha=nn.Parameter(torch.tensor(inverse_softplus(0.1),dtype=torch.float32))
    def forward(self,feat,base,mask):
        delta=self.net(feat).squeeze(-1)
        score=base+F.softplus(self.raw_alpha)*delta
        return score.masked_fill(~mask,-1e9)

@torch.no_grad()
def encode_rows(enc,x,batch=512):
    enc.eval(); out=[]
    for left in range(0,len(x),batch):
        t=torch.from_numpy(x[left:left+batch]).float().to(DEVICE)
        out.append(enc(t).cpu().numpy())
    return np.concatenate(out,axis=0).astype(np.float32)

def topk_np(score,k):
    k=min(int(k),len(score))
    idx=np.argpartition(-score,k-1)[:k]
    return idx[np.argsort(-score[idx])]

def pair_block(q,cand):
    qrep=np.repeat(q[None,:],len(cand),axis=0)
    return np.concatenate([qrep,cand,qrep-cand,np.abs(qrep-cand)],axis=1).astype(np.float32)

In [7]:
# Cell 6 — load only the frozen learned branch + feature scaler from current PRR
#
# Compatibility note:
# Some PRR checkpoints were serialized in an environment with NumPy >= 2,
# where pickle module paths use ``numpy._core.*``.  This notebook may run in
# Python 3.8 / NumPy 1.x, where the equivalent modules are ``numpy.core.*``.
# The custom unpickler below remaps only those NumPy module paths; the checkpoint
# contents themselves are unchanged.

import pickle
import types

class _NumpyCoreCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "numpy._core":
            module = "numpy.core"
        elif module.startswith("numpy._core."):
            module = "numpy.core." + module[len("numpy._core."):]
        return super().find_class(module, name)

def _compat_pickle_load(file_obj, **kwargs):
    return _NumpyCoreCompatUnpickler(file_obj, **kwargs).load()

_NUMPY_COMPAT_PICKLE = types.ModuleType("numpy_core_compat_pickle")
_NUMPY_COMPAT_PICKLE.Unpickler = _NumpyCoreCompatUnpickler
_NUMPY_COMPAT_PICKLE.load = _compat_pickle_load
_NUMPY_COMPAT_PICKLE.loads = pickle.loads
_NUMPY_COMPAT_PICKLE.dump = pickle.dump
_NUMPY_COMPAT_PICKLE.dumps = pickle.dumps

def _torch_load_standard(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        # Older torch versions do not expose weights_only.
        return torch.load(path, map_location=DEVICE)

def _torch_load_numpy_compat(path):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            pickle_module=_NUMPY_COMPAT_PICKLE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
            pickle_module=_NUMPY_COMPAT_PICKLE,
        )

def safe_torch_load(path):
    try:
        return _torch_load_standard(path)
    except ModuleNotFoundError as e:
        # Checkpoints written with NumPy 2.x can contain numpy._core.* paths.
        if e.name == "numpy._core" or (e.name and e.name.startswith("numpy._core.")):
            print(
                "[compat] checkpoint uses NumPy-2 pickle paths; "
                "remapping numpy._core.* -> numpy.core.*"
            )
            return _torch_load_numpy_compat(path)
        raise

class FrozenLearnedBranch:
    def __init__(self,qenc,henc,stats_med,stats_iqr,training_channels,training_query_anchors,source):
        self.qenc=qenc.eval(); self.henc=henc.eval()
        self.stats_med=np.asarray(stats_med,np.float32)
        self.stats_iqr=np.asarray(stats_iqr,np.float32)
        self.training_channels=np.asarray(training_channels,np.int64)
        self.training_query_anchors=np.asarray(training_query_anchors,np.int64)
        self.source=str(source)
    def scale_stats(self,x):
        return np.clip((x-self.stats_med)/self.stats_iqr,-cfg.feature_clip,cfg.feature_clip).astype(np.float32)

def load_frozen_branch(dataset,horizon):
    path=STACKS[(dataset,int(horizon))]
    obj=safe_torch_load(path)
    need={"query_encoder","history_encoder","stats_med","stats_iqr"}
    missing=need-set(obj)
    assert not missing,(path,missing)

    seed_everything(cfg.seed)
    qenc=TemporalEncoder1D(cfg.enc_hidden,cfg.emb_dim).to(DEVICE)
    henc=TemporalEncoder1D(cfg.enc_hidden,cfg.emb_dim).to(DEVICE)
    qenc.load_state_dict(obj["query_encoder"],strict=True)
    henc.load_state_dict(obj["history_encoder"],strict=True)

    return FrozenLearnedBranch(
        qenc,henc,obj["stats_med"],obj["stats_iqr"],
        obj.get("training_channels",[]),obj.get("training_query_anchors",[]),path
    )

# strict-load smoke test on one condition
_smoke=load_frozen_branch("ETTh1",96)
print("[PASS] frozen learned branch strict-load |",_smoke.source)
del _smoke
gc.collect(); torch.cuda.empty_cache()


[PASS] frozen learned branch strict-load | /data/code/which_histories_matter_ext/checkpoints/11_etth1_all_horizons_five_backbones_fixed_alpha01/full_stacks/ETTh1_H96_Full.pt


In [8]:
# Cell 7 — exact train-only geometry, preferring metadata stored in the existing PRR stack

def evenly_spaced_indices(n,k):
    n=int(n); k=min(int(k),n)
    if k>=n: return np.arange(n,dtype=np.int64)
    idx=np.unique(np.round(np.linspace(0,n-1,num=k)).astype(np.int64))
    if len(idx)<k:
        unused=np.setdiff1d(np.arange(n,dtype=np.int64),idx)
        idx=np.sort(np.concatenate([idx,unused[:k-len(idx)]]))
    return idx[:k]

def training_geometry(data,horizon,branch):
    h=int(horizon); train_end=int(data["train_end"])
    memory_end=int(cfg.train_memory_fraction*train_end)
    if data["name"]=="ETTh1":
        memory_end=max(memory_end,cfg.retrieval_seq_len+h+cfg.memory_stride)

    memory_anchors=np.arange(cfg.retrieval_seq_len,memory_end-h+1,cfg.memory_stride,dtype=np.int64)
    all_query=np.arange(memory_end,train_end-h+1,cfg.train_query_stride,dtype=np.int64)

    if len(branch.training_channels)>0:
        channels=branch.training_channels.copy()
    elif data["name"]=="ETTh1":
        channels=np.arange(data["n_channels"],dtype=np.int64)
    else:
        channels=evenly_spaced_indices(data["n_channels"],min(data["n_channels"],cfg.screen_max_train_channels))

    if len(branch.training_query_anchors)>0:
        qanchors=branch.training_query_anchors.copy()
    elif data["name"]=="ETTh1":
        qanchors=all_query
    else:
        max_q=max(4,cfg.screen_max_train_pairs//len(channels))
        qpos=evenly_spaced_indices(len(all_query),min(len(all_query),max_q))
        qanchors=all_query[qpos]

    assert np.all(qanchors>=memory_end)
    assert np.all(qanchors+h<=train_end)
    return dict(memory_end=memory_end,memory_anchors=memory_anchors,
                training_channels=channels,query_anchors=qanchors,
                training_pairs=len(channels)*len(qanchors))

def build_problem(data,horizon,branch):
    geom=training_geometry(data,horizon,branch)
    M=len(geom["memory_anchors"]); C=len(geom["training_channels"])
    assert M>=cfg.top_m_anchored and M>=cfg.top_m_learned

    mem_past=np.empty((C,M,cfg.retrieval_seq_len),np.float32)
    mem_future=np.empty((C,M,int(horizon)),np.float32)
    mem_stats=np.empty((C,M,7),np.float32)

    for slot,ch in enumerate(geom["training_channels"]):
        p,f=extract_channel(data["z"],int(ch),geom["memory_anchors"],horizon)
        mem_past[slot]=p; mem_future[slot]=f
        mem_stats[slot]=branch.scale_stats(context7(p))

    return {**geom,"M":M,"C":C,"horizon":int(horizon),
            "mem_past":mem_past,"mem_future":mem_future,"mem_stats":mem_stats}

In [9]:
# Cell 8 — build Anchored-100 ∪ Learned-100 future-supervised reranker records

def build_aprr_records(data,problem,branch):
    qanchors=problem["query_anchors"]
    C=problem["C"]

    mem_emb=np.empty((C,problem["M"],cfg.emb_dim),np.float32)
    for slot in range(C):
        mem_emb[slot]=encode_rows(branch.henc,problem["mem_past"][slot])

    records=[]
    support_rows=[]

    for slot,ch in enumerate(problem["training_channels"]):
        qpast,qfuture=extract_channel(data["z"],int(ch),qanchors,problem["horizon"])
        qemb=encode_rows(branch.qenc,qpast)

        # Exact Anchored-L2 ranking, represented by a query-wise z-scored negative MSE.
        abase_all=anchored_similarity_z(qpast,problem["mem_past"][slot])
        learned_all=qemb@mem_emb[slot].T

        for j,anchor in enumerate(qanchors):
            atop=topk_np(abase_all[j],cfg.top_m_anchored)
            ltop=topk_np(learned_all[j],cfg.top_m_learned)
            union=np.union1d(atop,ltop).astype(np.int64)
            assert len(union)<=cfg.union_max

            dist=np.mean((problem["mem_future"][slot,union]-qfuture[j][None,:])**2,axis=1).astype(np.float32)
            qstat=branch.scale_stats(context7(qpast[j:j+1]))[0]

            records.append(dict(
                anchor=int(anchor),channel=int(ch),channel_slot=int(slot),cand_idx=union,
                base_score=abase_all[j,union].astype(np.float32),future_dist=dist,
                qstat=qstat.astype(np.float32),qemb=qemb[j].astype(np.float32),
            ))
            support_rows.append((len(union),len(np.intersect1d(atop,ltop)),len(np.setdiff1d(ltop,atop))))

    records.sort(key=lambda r:(r["anchor"],r["channel"]))
    unique_a=np.unique([r["anchor"] for r in records])
    cut_n=max(1,min(len(unique_a)-1,int(cfg.phase_a_train_fraction*len(unique_a))))
    cut_anchor=unique_a[cut_n]
    train_records=[r for r in records if r["anchor"]<cut_anchor]
    valid_records=[r for r in records if r["anchor"]>=cut_anchor]

    s=np.asarray(support_rows,float)
    support_summary=dict(mean_union_size=float(s[:,0].mean()),mean_overlap=float(s[:,1].mean()),
                         mean_learned_only=float(s[:,2].mean()))
    return records,train_records,valid_records,mem_emb,support_summary

In [10]:
# Cell 9 — reranker dataset and exact hybrid objective
class RerankDataset(Dataset):
    def __init__(self,records,problem,mem_emb):
        self.records=records; self.problem=problem; self.mem_emb=mem_emb
    def __len__(self): return len(self.records)
    def __getitem__(self,idx):
        r=self.records[idx]; slot=r["channel_slot"]; ci=r["cand_idx"]
        qjoint=np.concatenate([r["qstat"],r["qemb"]],axis=0)
        cjoint=np.concatenate([self.problem["mem_stats"][slot,ci],self.mem_emb[slot,ci]],axis=1)
        feat=np.concatenate([r["base_score"][:,None],pair_block(qjoint,cjoint)],axis=1).astype(np.float32)
        n,D=feat.shape
        feat_pad=np.zeros((cfg.union_max,D),np.float32)
        base_pad=np.zeros(cfg.union_max,np.float32)
        dist_pad=np.zeros(cfg.union_max,np.float32)
        mask=np.zeros(cfg.union_max,bool)
        feat_pad[:n]=feat; base_pad[:n]=r["base_score"]; dist_pad[:n]=r["future_dist"]; mask[:n]=True
        return torch.from_numpy(feat_pad),torch.from_numpy(base_pad),torch.from_numpy(dist_pad),torch.from_numpy(mask)

def teacher_from_distance(dist,mask):
    m=mask.float(); n=m.sum(1,keepdim=True).clamp_min(1.0)
    mu=(dist*m).sum(1,keepdim=True)/n
    var=(((dist-mu)**2)*m).sum(1,keepdim=True)/n
    sd=torch.sqrt(var+1e-6)
    logits=-((dist-mu)/sd)/cfg.teacher_tau
    logits=logits.masked_fill(~mask,-1e9)
    return F.softmax(logits,dim=1)

def hybrid_loss(score,dist,mask):
    target=teacher_from_distance(dist,mask)
    listwise=-(target*F.log_softmax(score,dim=1)).sum(1).mean()
    safe=dist.masked_fill(~mask,float("inf"))
    pos_idx=torch.topk(safe,k=cfg.top_k,dim=1,largest=False).indices
    top_mask=torch.zeros_like(mask); top_mask.scatter_(1,pos_idx,True); top_mask&=mask
    neg_mask=mask&(~top_mask)
    pos_score=torch.gather(score,1,pos_idx)
    margin=pos_score.unsqueeze(2)-score.unsqueeze(1)
    pair_mask=neg_mask.unsqueeze(1).expand_as(margin)
    ploss=F.softplus(-margin)*pair_mask.float()
    denom=pair_mask.float().sum((1,2)).clamp_min(1.0)
    pairwise=(ploss.sum((1,2))/denom).mean()
    return listwise+cfg.pairwise_lambda*pairwise

@torch.no_grad()
def rerank_val_afmse(model,loader):
    model.eval(); vals=[]
    for feat,base,dist,mask in loader:
        feat=feat.float().to(DEVICE); base=base.float().to(DEVICE); dist=dist.float().to(DEVICE); mask=mask.to(DEVICE)
        score=model(feat,base,mask)
        top=torch.topk(score,k=cfg.top_k,dim=1).indices
        vals.append(torch.gather(dist,1,top).mean(1).cpu().numpy())
    return float(np.concatenate(vals).mean())

def make_reranker(seed):
    seed_everything(seed)
    return WideFusionReranker().to(DEVICE)

In [11]:
# Cell 10 — train one A-PRR reranker (frozen encoders; fresh reranker)

def reranker_path(dataset,horizon):
    return CKPT_ROOT/f"{dataset}_H{int(horizon)}_anchored_prr_reranker.pt"

def train_one_aprr(dataset,horizon,overwrite=False):
    out=reranker_path(dataset,horizon)
    if out.is_file() and not overwrite:
        print("[SKIP existing]",out)
        return out

    print("\n"+"="*100)
    print("TRAIN A-PRR RERANKER |",dataset,"H=",horizon)
    print("="*100)
    branch=load_frozen_branch(dataset,horizon)
    problem=build_problem(DATA[dataset],horizon,branch)
    all_rec,tr_rec,va_rec,mem_emb,support=build_aprr_records(DATA[dataset],problem,branch)

    tr_ds=RerankDataset(tr_rec,problem,mem_emb); va_ds=RerankDataset(va_rec,problem,mem_emb)
    g=torch.Generator().manual_seed(cfg.seed)
    tr_loader=DataLoader(tr_ds,batch_size=cfg.rerank_batch,shuffle=True,generator=g,num_workers=0)
    va_loader=DataLoader(va_ds,batch_size=cfg.rerank_batch,shuffle=False,num_workers=0)

    model=make_reranker(cfg.seed)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.rerank_lr,weight_decay=cfg.rerank_wd)
    best_val=float("inf"); best_epoch=1; wait=0; hist=[]

    for epoch in range(1,cfg.rerank_max_epochs+1):
        model.train(); losses=[]
        for feat,base,dist,mask in tr_loader:
            feat=feat.float().to(DEVICE); base=base.float().to(DEVICE); dist=dist.float().to(DEVICE); mask=mask.to(DEVICE)
            score=model(feat,base,mask); loss=hybrid_loss(score,dist,mask)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip); opt.step()
            losses.append(float(loss.item()))
        val=rerank_val_afmse(model,va_loader)
        hist.append(dict(epoch=epoch,train_loss=float(np.mean(losses)),val_analog_future_mse=val))
        print(f"[A-PRR {dataset} H={horizon}] ep={epoch:02d} valAFMSE={val:.6f}")
        if val<best_val-1e-8:
            best_val=val; best_epoch=epoch; wait=0
        else:
            wait+=1
            if wait>=cfg.rerank_patience: break

    pd.DataFrame(hist).to_csv(OUT_ROOT/f"{dataset}_H{horizon}_phaseA.csv",index=False)

    # Fresh initialization; refit all train-only records for validation-selected epoch count.
    all_ds=RerankDataset(all_rec,problem,mem_emb)
    g2=torch.Generator().manual_seed(cfg.seed)
    all_loader=DataLoader(all_ds,batch_size=cfg.rerank_batch,shuffle=True,generator=g2,num_workers=0)
    model=make_reranker(cfg.seed)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.rerank_lr,weight_decay=cfg.rerank_wd)
    for epoch in range(1,best_epoch+1):
        model.train()
        for feat,base,dist,mask in all_loader:
            feat=feat.float().to(DEVICE); base=base.float().to(DEVICE); dist=dist.float().to(DEVICE); mask=mask.to(DEVICE)
            loss=hybrid_loss(model(feat,base,mask),dist,mask)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip); opt.step()
        print(f"[A-PRR refit {dataset} H={horizon}] {epoch:02d}/{best_epoch}")

    torch.save(dict(
        reranker=model.state_dict(),dataset=dataset,horizon=int(horizon),best_epoch=int(best_epoch),
        frozen_learned_stack=str(branch.source),base_retriever="last_value_anchored_l2_top100",
        base_score="query_zscored_negative_anchored_mse",
        learned_retriever="frozen_full_prr_future_supervised_encoder_top100",
        support_summary=support,cfg=cfg.__dict__,
    ),out)
    print("[SAVED]",out,"|",support)

    del branch,problem,all_rec,tr_rec,va_rec,mem_emb,tr_ds,va_ds,all_ds,tr_loader,va_loader,all_loader,model
    gc.collect(); torch.cuda.empty_cache()
    return out

In [12]:
# Cell 11 — train/resume the selected A-PRR rerankers
if RUN_TRAINING:
    for d,h in TARGETS:
        train_one_aprr(d,h,overwrite=OVERWRITE_RERANKER)
else:
    print("RUN_TRAINING=False")

inventory=[]
for d,h in TARGETS:
    p=reranker_path(d,h)
    inventory.append(dict(dataset=d,horizon=h,exists=p.is_file(),size_MB=p.stat().st_size/2**20 if p.is_file() else np.nan,path=str(p)))
reranker_inventory=pd.DataFrame(inventory)
display(reranker_inventory)
print("A-PRR rerankers:",int(reranker_inventory.exists.sum()),"/",len(reranker_inventory))


TRAIN A-PRR RERANKER | ETTh1 H= 96
[A-PRR ETTh1 H=96] ep=01 valAFMSE=0.596203
[A-PRR ETTh1 H=96] ep=02 valAFMSE=0.576258
[A-PRR ETTh1 H=96] ep=03 valAFMSE=0.567120
[A-PRR ETTh1 H=96] ep=04 valAFMSE=0.573326
[A-PRR ETTh1 H=96] ep=05 valAFMSE=0.570338
[A-PRR ETTh1 H=96] ep=06 valAFMSE=0.563750
[A-PRR ETTh1 H=96] ep=07 valAFMSE=0.564886
[A-PRR ETTh1 H=96] ep=08 valAFMSE=0.568075
[A-PRR ETTh1 H=96] ep=09 valAFMSE=0.569397
[A-PRR ETTh1 H=96] ep=10 valAFMSE=0.566140
[A-PRR ETTh1 H=96] ep=11 valAFMSE=0.563598
[A-PRR ETTh1 H=96] ep=12 valAFMSE=0.564267
[A-PRR ETTh1 H=96] ep=13 valAFMSE=0.564538
[A-PRR ETTh1 H=96] ep=14 valAFMSE=0.566337
[A-PRR ETTh1 H=96] ep=15 valAFMSE=0.561038
[A-PRR ETTh1 H=96] ep=16 valAFMSE=0.568621
[A-PRR ETTh1 H=96] ep=17 valAFMSE=0.569151
[A-PRR ETTh1 H=96] ep=18 valAFMSE=0.565987
[A-PRR ETTh1 H=96] ep=19 valAFMSE=0.568637
[A-PRR ETTh1 H=96] ep=20 valAFMSE=0.566138
[A-PRR ETTh1 H=96] ep=21 valAFMSE=0.566168
[A-PRR refit ETTh1 H=96] 01/15
[A-PRR refit ETTh1 H=96] 02/15

,dataset,horizon,exists,size_MB,path
0,ETTh1,96,True,0.996180,/data/code/which_histories_matter_ext/results/...
1,ETTh1,192,True,0.996257,/data/code/which_histories_matter_ext/results/...
2,ETTh1,336,True,0.996257,/data/code/which_histories_matter_ext/results/...
3,ETTh1,720,True,0.996257,/data/code/which_histories_matter_ext/results/...
4,Weather,96,True,0.996273,/data/code/which_histories_matter_ext/results/...
5,Weather,192,True,0.996289,/data/code/which_histories_matter_ext/results/...
6,Weather,336,True,0.996289,/data/code/which_histories_matter_ext/results/...
7,Weather,720,True,0.996289,/data/code/which_histories_matter_ext/results/...
8,Electricity,96,True,0.996338,/data/code/which_histories_matter_ext/results/...
9,Electricity,192,True,0.996354,/data/code/which_histories_matter_ext/results/...


A-PRR rerankers: 24 / 24


In [13]:
# Cell 12 — exact frozen test manifests and A-PRR evaluation helpers

def load_manifest(dataset,horizon,split="test"):
    p=MANIFEST_ROOT/f"{dataset}_H{int(horizon)}_{split}.npz"
    assert p.exists(),p
    with np.load(p,allow_pickle=False) as z:
        a=np.asarray(z["anchor"],np.int64); c=np.asarray(z["channel"],np.int64)
    assert a.shape==c.shape and len(a)>0
    return a,c

def load_aprr_reranker(dataset,horizon):
    p=reranker_path(dataset,horizon); assert p.exists(),p
    obj=safe_torch_load(p)
    model=make_reranker(cfg.seed)
    model.load_state_dict(obj["reranker"],strict=True); model.eval()
    return model,obj

def test_part_path(dataset,horizon,ch):
    p=CACHE_ROOT/dataset/f"H{int(horizon)}"
    p.mkdir(parents=True,exist_ok=True)
    return p/f"ch{int(ch):04d}.npz"

@torch.no_grad()
def evaluate_channel(dataset,horizon,branch,reranker,boundary,qa,ch):
    z=DATA[dataset]["z"]; h=int(horizon)
    mem_anchor=np.arange(cfg.retrieval_seq_len,int(boundary)-h+1,cfg.memory_stride,dtype=np.int64)
    assert len(mem_anchor)>=max(cfg.top_m_anchored,cfg.top_m_learned)
    mp,mf=extract_channel(z,int(ch),mem_anchor,h)
    mem_stats=branch.scale_stats(context7(mp))
    mem_emb=encode_rows(branch.henc,mp)

    qp,qf=extract_channel(z,int(ch),qa,h)
    qstats=branch.scale_stats(context7(qp)); qemb=encode_rows(branch.qenc,qp)
    abase=anchored_similarity_z(qp,mp)
    lsim=qemb@mem_emb.T

    analog=np.empty(len(qa),np.float32)
    retr_mse=np.empty(len(qa),np.float32)
    union_size=np.empty(len(qa),np.int16)
    overlap=np.empty(len(qa),np.int16)
    learned_only=np.empty(len(qa),np.int16)
    selected_learned_only=np.empty(len(qa),np.float32)

    for left in range(0,len(qa),cfg.query_batch):
        right=min(len(qa),left+cfg.query_batch); B=right-left
        cand=np.zeros((B,cfg.union_max),np.int64); valid=np.zeros((B,cfg.union_max),bool)
        lo_masks=[]
        for j in range(B):
            row=left+j
            atop=topk_np(abase[row],cfg.top_m_anchored)
            ltop=topk_np(lsim[row],cfg.top_m_learned)
            u=np.union1d(atop,ltop).astype(np.int64)
            lo=np.setdiff1d(ltop,atop)
            n=len(u); cand[j,:n]=u; valid[j,:n]=True
            union_size[row]=n; overlap[row]=len(np.intersect1d(atop,ltop)); learned_only[row]=len(lo)
            lo_masks.append(set(lo.tolist()))

        bscore=np.take_along_axis(abase[left:right],cand,axis=1).astype(np.float32)
        cstats=mem_stats[cand]; cemb=mem_emb[cand]
        qjoint=np.concatenate([qstats[left:right],qemb[left:right]],axis=1)
        cjoint=np.concatenate([cstats,cemb],axis=2)
        qrep=np.repeat(qjoint[:,None,:],cfg.union_max,axis=1)
        feat=np.concatenate([bscore[:,:,None],qrep,cjoint,qrep-cjoint,np.abs(qrep-cjoint)],axis=2).astype(np.float32)
        score=reranker(torch.from_numpy(feat).float().to(DEVICE),
                       torch.from_numpy(bscore).float().to(DEVICE),
                       torch.from_numpy(valid).to(DEVICE)).cpu().numpy()
        top=np.argsort(-score,axis=1)[:,:cfg.top_k]
        selected=np.take_along_axis(cand,top,axis=1)

        chosen=mf[selected]
        analog[left:right]=((chosen-qf[left:right,None,:])**2).mean(axis=(1,2)).astype(np.float32)
        mean_rel=chosen.mean(axis=1)
        retr_mse[left:right]=((mean_rel-qf[left:right])**2).mean(axis=1).astype(np.float32)
        for j in range(B):
            selected_learned_only[left+j]=sum(int(x) in lo_masks[j] for x in selected[j])/cfg.top_k

    return dict(analog=analog,retrieval_forecast_mse=retr_mse,union_size=union_size,
                overlap=overlap,learned_only=learned_only,selected_learned_only=selected_learned_only)

In [14]:
# Cell 13 — run exact 24-condition TEST evaluation, resumably
rows=[]
if RUN_EVALUATION:
    for d,h in TARGETS:
        print("\n"+"="*100); print("TEST A-PRR |",d,"H=",h); print("="*100)
        qa,qc=load_manifest(d,h,"test")
        boundary=DATA[d]["val_end"]
        branch=load_frozen_branch(d,h)
        reranker,meta=load_aprr_reranker(d,h)

        analog=np.full(len(qa),np.nan,np.float32)
        rfmse=np.full(len(qa),np.nan,np.float32)
        union=np.zeros(len(qa),np.int16); overlap=np.zeros(len(qa),np.int16); lo=np.zeros(len(qa),np.int16)
        sel_lo=np.zeros(len(qa),np.float32)
        t0=time.perf_counter()

        for ch in np.unique(qc):
            idx=np.flatnonzero(qc==ch); aa=qa[idx]; pp=test_part_path(d,h,ch)
            use_cache=False
            if pp.exists() and not OVERWRITE_TEST_CACHE:
                try:
                    with np.load(pp,allow_pickle=False) as zc:
                        use_cache=np.array_equal(zc["anchor"],aa)
                        if use_cache:
                            analog[idx]=zc["analog"]; rfmse[idx]=zc["retrieval_forecast_mse"]
                            union[idx]=zc["union_size"]; overlap[idx]=zc["overlap"]
                            lo[idx]=zc["learned_only"]; sel_lo[idx]=zc["selected_learned_only"]
                except Exception:
                    use_cache=False
            if not use_cache:
                out=evaluate_channel(d,h,branch,reranker,boundary,aa,int(ch))
                analog[idx]=out["analog"]; rfmse[idx]=out["retrieval_forecast_mse"]
                union[idx]=out["union_size"]; overlap[idx]=out["overlap"]
                lo[idx]=out["learned_only"]; sel_lo[idx]=out["selected_learned_only"]
                np.savez_compressed(pp,anchor=aa,analog=out["analog"],retrieval_forecast_mse=out["retrieval_forecast_mse"],
                                    union_size=out["union_size"],overlap=out["overlap"],learned_only=out["learned_only"],
                                    selected_learned_only=out["selected_learned_only"])
            print(f"  ch={int(ch):4d} pairs={len(idx):4d} {'cache' if use_cache else 'done'}")

        assert np.isfinite(analog).all() and np.isfinite(rfmse).all()
        row=dict(dataset=d,horizon=int(h),pairs=len(qa),
                 aprr_analog_future_mse=float(analog.mean()),
                 aprr_retrieval_forecast_mse=float(rfmse.mean()),
                 mean_union_size=float(union.mean()),mean_overlap=float(overlap.mean()),
                 mean_learned_only=float(lo.mean()),
                 mean_final_top10_learned_only_fraction=float(sel_lo.mean()),
                 rerank_best_epoch=int(meta["best_epoch"]),seconds=float(time.perf_counter()-t0))
        rows.append(row); print(row)
        pd.DataFrame(rows).to_csv(OUT_ROOT/"aprr_test24_running.csv",index=False)
        del branch,reranker; gc.collect(); torch.cuda.empty_cache()

    aprr24=pd.DataFrame(rows).sort_values(["dataset","horizon"]).reset_index(drop=True)
    assert len(aprr24)==len(TARGETS)
    aprr24.to_csv(OUT_ROOT/"aprr_test24.csv",index=False)
else:
    p=OUT_ROOT/"aprr_test24.csv"
    assert p.exists(),p
    aprr24=pd.read_csv(p)

display(aprr24)


TEST A-PRR | ETTh1 H= 96
  ch=   0 pairs=1170 done
  ch=   1 pairs=1170 done
  ch=   2 pairs=1170 done
  ch=   3 pairs=1170 done
  ch=   4 pairs=1170 done
  ch=   5 pairs=1171 done
  ch=   6 pairs=1171 done
{'dataset': 'ETTh1', 'horizon': 96, 'pairs': 8192, 'aprr_analog_future_mse': 1.169513463973999, 'aprr_retrieval_forecast_mse': 0.8441636562347412, 'mean_union_size': 178.6600341796875, 'mean_overlap': 21.3399658203125, 'mean_learned_only': 78.6600341796875, 'mean_final_top10_learned_only_fraction': 0.610827624797821, 'rerank_best_epoch': 15, 'seconds': 3.156196191906929}

TEST A-PRR | ETTh1 H= 192
  ch=   0 pairs=1170 done
  ch=   1 pairs=1170 done
  ch=   2 pairs=1170 done
  ch=   3 pairs=1170 done
  ch=   4 pairs=1170 done
  ch=   5 pairs=1171 done
  ch=   6 pairs=1171 done
{'dataset': 'ETTh1', 'horizon': 192, 'pairs': 8192, 'aprr_analog_future_mse': 1.194267749786377, 'aprr_retrieval_forecast_mse': 0.883638858795166, 'mean_union_size': 179.5584716796875, 'mean_overlap': 20.44152

,dataset,horizon,pairs,aprr_analog_future_mse,aprr_retrieval_forecast_mse,mean_union_size,mean_overlap,mean_learned_only,mean_final_top10_learned_only_fraction,rerank_best_epoch,seconds
0,ETTh1,96,8192,1.169513,0.844164,178.660034,21.339966,78.660034,0.610828,15,3.156196
1,ETTh1,192,8192,1.194268,0.883639,179.558472,20.441528,79.558472,0.651917,7,3.097629
2,ETTh1,336,8192,1.438145,1.007252,182.452759,17.547241,82.452759,0.673450,11,3.142277
3,ETTh1,720,8192,1.446335,1.113739,183.043335,16.956665,83.043335,0.717859,18,3.227344
4,Electricity,96,8192,0.716782,0.551696,157.494507,42.505493,57.494507,0.190649,24,7.039082
5,Electricity,192,8192,0.733713,0.559901,156.397095,43.602905,56.397095,0.146582,22,7.586636
6,Electricity,336,8192,0.774582,0.581737,159.852905,40.147095,59.852905,0.168384,25,8.489673
7,Electricity,720,8192,0.858952,0.636148,167.626709,32.373291,67.626709,0.258435,15,10.394761
8,Exchange,96,8192,0.199907,0.113323,185.393066,14.606934,85.393066,0.789258,6,3.036987
9,Exchange,192,8192,0.411132,0.267099,185.572144,14.427856,85.572144,0.746313,3,3.028130


In [15]:
# Cell 14 — load exact Anchored-L2 / Pearson / current-PRR references
# Prefer generated local CSVs; otherwise use the already frozen paper-facing references.

baseline = None
l2_csv = PROJECT_ROOT/"results"/"anchored_l2_long_horizon_frozen24"/"anchored_l2_test_24conditions.csv"
precise_csv = PROJECT_ROOT/"results"/"stride8_revalidation"/"table1_stride8_revalidated.csv"

if l2_csv.exists():
    l2=pd.read_csv(l2_csv)[["dataset","horizon","anchored_l2_analog_future_mse"]].rename(
        columns={"anchored_l2_analog_future_mse":"anchored_l2"})
else:
    l2 = pd.DataFrame([
        ("ETTh1",96,1.100985),("ETTh1",192,1.277228),("ETTh1",336,1.446432),("ETTh1",720,1.714012),
        ("Electricity",96,.703011),("Electricity",192,.728439),("Electricity",336,.768702),("Electricity",720,.815970),
        ("Exchange",96,.186254),("Exchange",192,.379211),("Exchange",336,.684537),("Exchange",720,1.865029),
        ("Solar",96,.488054),("Solar",192,.518212),("Solar",336,.533977),("Solar",720,.527471),
        ("Traffic",96,1.420632),("Traffic",192,1.355879),("Traffic",336,1.424405),("Traffic",720,1.595257),
        ("Weather",96,.839593),("Weather",192,.785080),("Weather",336,.594079),("Weather",720,.769931),
    ],columns=["dataset","horizon","anchored_l2"])

if precise_csv.exists():
    p=pd.read_csv(precise_csv)
    need={"dataset","horizon","pearson_analog_future_mse","prr_analog_future_mse"}
    assert need.issubset(p.columns),p.columns
    ref=p[["dataset","horizon","pearson_analog_future_mse","prr_analog_future_mse"]].rename(
        columns={"pearson_analog_future_mse":"pearson","prr_analog_future_mse":"prr"})
else:
    ref=pd.DataFrame([
        ("ETTh1",96,1.612267,1.1643),("ETTh1",192,1.817531,1.2229),("ETTh1",336,1.992872,1.4816),("ETTh1",720,2.254092,1.4618),
        ("Weather",96,1.161329,.4316),("Weather",192,1.062809,.5147),("Weather",336,1.290959,.6748),("Weather",720,1.295817,.8130),
        ("Electricity",96,1.179589,.7191),("Electricity",192,1.195698,.7345),("Electricity",336,1.241844,.7648),("Electricity",720,1.287729,.8602),
        ("Traffic",96,1.973787,1.4696),("Traffic",192,2.004524,1.4508),("Traffic",336,2.063635,1.5147),("Traffic",720,2.166454,1.7351),
        ("Exchange",96,.213266,.1652),("Exchange",192,.423408,.4116),("Exchange",336,.764282,.8094),("Exchange",720,1.886809,1.7770),
        ("Solar",96,.588674,.5087),("Solar",192,.597890,.5406),("Solar",336,.623836,.5701),("Solar",720,.638530,.5507),
    ],columns=["dataset","horizon","pearson","prr"])

baseline=ref.merge(l2,on=["dataset","horizon"],how="inner")
baseline["horizon"]=baseline["horizon"].astype(int)
display(baseline.sort_values(["dataset","horizon"]))

,dataset,horizon,pearson,prr,anchored_l2
0,ETTh1,96,1.612267,1.1643,1.100985
1,ETTh1,192,1.817531,1.2229,1.277228
2,ETTh1,336,1.992872,1.4816,1.446432
3,ETTh1,720,2.254092,1.4618,1.714012
4,Electricity,96,1.179589,0.7191,0.703011
5,Electricity,192,1.195698,0.7345,0.728439
6,Electricity,336,1.241844,0.7648,0.768702
7,Electricity,720,1.287729,0.8602,0.815970
8,Exchange,96,0.213266,0.1652,0.186254
9,Exchange,192,0.423408,0.4116,0.379211


In [16]:
# Cell 15 — exact strong-base comparisons and paper-facing summary
comp=(aprr24.merge(baseline,on=["dataset","horizon"],how="inner")
      .sort_values(["dataset","horizon"]).reset_index(drop=True))
assert len(comp)==len(TARGETS)
comp["aprr"]=comp["aprr_analog_future_mse"]

# Positive = A-PRR has lower MSE than the named baseline.
comp["aprr_gain_vs_l2_pct"]=100*(comp["anchored_l2"]-comp["aprr"])/comp["anchored_l2"]
comp["aprr_gain_vs_prr_pct"]=100*(comp["prr"]-comp["aprr"])/comp["prr"]
comp["aprr_gain_vs_pearson_pct"]=100*(comp["pearson"]-comp["aprr"])/comp["pearson"]


def wtl(x,tol=1e-12):
    x=np.asarray(x,float)
    return dict(W=int((x>tol).sum()),T=int((np.abs(x)<=tol).sum()),L=int((x<-tol).sum()),
                mean_gain_pct=float(x.mean()),median_gain_pct=float(np.median(x)))

summary=pd.DataFrame([
    {"comparison":"A-PRR vs Anchored L2",**wtl(comp.aprr_gain_vs_l2_pct)},
    {"comparison":"A-PRR vs current PRR",**wtl(comp.aprr_gain_vs_prr_pct)},
    {"comparison":"A-PRR vs Pearson",**wtl(comp.aprr_gain_vs_pearson_pct)},
])

dataset_rows=[]
for d,g in comp.groupby("dataset",sort=False):
    a=wtl(g.aprr_gain_vs_l2_pct); b=wtl(g.aprr_gain_vs_prr_pct)
    dataset_rows.append(dict(dataset=d,
        Aprr_vs_L2_W=a["W"],Aprr_vs_L2_T=a["T"],Aprr_vs_L2_L=a["L"],Aprr_vs_L2_mean_pct=a["mean_gain_pct"],
        Aprr_vs_PRR_W=b["W"],Aprr_vs_PRR_T=b["T"],Aprr_vs_PRR_L=b["L"],Aprr_vs_PRR_mean_pct=b["mean_gain_pct"],
        mean_union_size=float(g.mean_union_size.mean()),mean_learned_only=float(g.mean_learned_only.mean()),
        final_top10_learned_only_pct=100*float(g.mean_final_top10_learned_only_fraction.mean())))
dataset_summary=pd.DataFrame(dataset_rows)

display(comp[["dataset","horizon","pearson","anchored_l2","prr","aprr",
              "aprr_gain_vs_l2_pct","aprr_gain_vs_prr_pct","mean_union_size","mean_learned_only",
              "mean_final_top10_learned_only_fraction"]])
display(summary)
display(dataset_summary)

comp.to_csv(OUT_ROOT/"exact_aprr_vs_l2_prr_24.csv",index=False)
summary.to_csv(OUT_ROOT/"aprr_overall_summary.csv",index=False)
dataset_summary.to_csv(OUT_ROOT/"aprr_dataset_summary.csv",index=False)

print("\n"+"="*100)
print("PAPER-FACING STRONG-BASE RESULT")
print("="*100)
for r in summary.itertuples(index=False):
    print(f"{r.comparison}: {r.W}W/{r.T}T/{r.L}L | mean={r.mean_gain_pct:+.3f}% | median={r.median_gain_pct:+.3f}%")
print(f"Mean A-PRR union size: {comp.mean_union_size.mean():.1f}")
print(f"Mean learned-only candidates beyond Anchored Top-100: {comp.mean_learned_only.mean():.1f}")
print(f"Mean final Top-10 learned-only fraction: {100*comp.mean_final_top10_learned_only_fraction.mean():.1f}%")
print("Saved under:",OUT_ROOT)

,dataset,horizon,pearson,anchored_l2,prr,aprr,aprr_gain_vs_l2_pct,aprr_gain_vs_prr_pct,mean_union_size,mean_learned_only,mean_final_top10_learned_only_fraction
0,ETTh1,96,1.612267,1.100985,1.1643,1.169513,-6.224259,-0.447777,178.660034,78.660034,0.610828
1,ETTh1,192,1.817531,1.277228,1.2229,1.194268,6.495362,2.341340,179.558472,79.558472,0.651917
2,ETTh1,336,1.992872,1.446432,1.4816,1.438145,0.572915,2.932959,182.452759,82.452759,0.673450
3,ETTh1,720,2.254092,1.714012,1.4618,1.446335,15.616990,1.057953,183.043335,83.043335,0.717859
4,Electricity,96,1.179589,0.703011,0.7191,0.716782,-1.958886,0.322384,157.494507,57.494507,0.190649
5,Electricity,192,1.195698,0.728439,0.7345,0.733713,-0.723915,0.107192,156.397095,56.397095,0.146582
6,Electricity,336,1.241844,0.768702,0.7648,0.774582,-0.764941,-1.279046,159.852905,59.852905,0.168384
7,Electricity,720,1.287729,0.815970,0.8602,0.858952,-5.267610,0.145105,167.626709,67.626709,0.258435
8,Exchange,96,0.213266,0.186254,0.1652,0.199907,-7.330289,-21.009263,185.393066,85.393066,0.789258
9,Exchange,192,0.423408,0.379211,0.4116,0.411132,-8.417648,0.113708,185.572144,85.572144,0.746313


,comparison,W,T,L,mean_gain_pct,median_gain_pct
0,A-PRR vs Anchored L2,6,0,18,1.633741,-2.385494
1,A-PRR vs current PRR,19,0,5,0.839667,1.374056
2,A-PRR vs Pearson,24,0,0,27.512152,27.932370


,dataset,Aprr_vs_L2_W,Aprr_vs_L2_T,Aprr_vs_L2_L,Aprr_vs_L2_mean_pct,Aprr_vs_PRR_W,Aprr_vs_PRR_T,Aprr_vs_PRR_L,Aprr_vs_PRR_mean_pct,mean_union_size,mean_learned_only,final_top10_learned_only_pct
0,ETTh1,3,0,1,4.115252,3,0,1,1.471119,180.928650,80.928650,66.351317
1,Electricity,0,0,4,-2.178838,3,0,1,-0.176091,160.342804,60.342804,19.101258
2,Exchange,1,0,3,-2.569892,3,0,1,-0.958312,185.533563,85.533563,78.921138
3,Solar,0,0,4,-4.861663,3,0,1,0.065871,168.758789,68.758789,36.864014
4,Traffic,0,0,4,-2.898147,4,0,0,3.258720,158.287445,58.287445,21.001893
5,Weather,2,0,2,18.195733,3,0,1,1.376694,189.182709,89.182709,48.864440



PAPER-FACING STRONG-BASE RESULT
A-PRR vs Anchored L2: 6W/0T/18L | mean=+1.634% | median=-2.385%
A-PRR vs current PRR: 19W/0T/5L | mean=+0.840% | median=+1.374%
A-PRR vs Pearson: 24W/0T/0L | mean=+27.512% | median=+27.932%
Mean A-PRR union size: 173.8
Mean learned-only candidates beyond Anchored Top-100: 73.8
Mean final Top-10 learned-only fraction: 45.2%
Saved under: /data/code/which_histories_matter_ext/results/exp46_anchored_prr_strong_base


## How to interpret the result

The key line is **`A-PRR vs Anchored L2`**.

- If A-PRR improves Anchored L2 broadly (especially with a positive mean/median and wins across several datasets), the paper can present PRR more generally as a **base-retriever + learned support expansion + future-supervised reranking framework**, rather than as a method tied to Pearson.
- If A-PRR is only near-tied with Anchored L2, the result is still informative: the handcrafted endpoint-relative inductive bias already captures much of the useful retrieval structure.
- If A-PRR is worse, keep Anchored L2 as the strong deterministic diagnostic in Appendix G and do **not** promote A-PRR into the main method.

The `mean_final_top10_learned_only_fraction` diagnostic is especially useful: it measures how often the final future-supervised reranker actually selects candidates that the learned branch contributes **outside Anchored Top-100**. This provides direct evidence about whether learned support expansion remains useful after starting from a much stronger deterministic retriever.